In [1]:
%%writefile main.py
"""
Orbit Wars - "Interstellar Monopoly: Asset Liquidation" Agent

Upgraded Strategy:
  1. Comet Path Prediction: Intercepts comets using their exact trajectory arrays.
  2. Profit Margin Targeting: Only buys comets if (Remaining Lifespan > Acquisition Cost).
  3. "Pump and Dump" (Evacuation): Siphons ships off comets right before they leave the 
     board, sending the capital to safe permanent planets.
  4. Standard Monopoly Logic: Avoids the sun, predicts orbits, prevents double-spending.
"""

import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

def get_speed(ships):
    if ships <= 1: 
        return 1.0
    ratio = min(1.0, max(0.0, math.log(ships) / math.log(1000)))
    return 1.0 + 5.0 * (ratio ** 1.5)

def intersects_sun(x1, y1, x2, y2, margin=10.5):
    px, py = 50.0, 50.0
    l2 = (x2 - x1)**2 + (y2 - y1)**2
    if l2 == 0: return False
    t = max(0, min(1, ((px - x1)*(x2 - x1) + (py - y1)*(y2 - y1)) / l2))
    return math.hypot(px - (x1 + t*(x2 - x1)), py - (y1 + t*(y2 - y1))) <= margin

def is_under_acquisition(target, my_fleets, angular_velocity, comet_data):
    """Checks if we already have a fleet perfectly intercepted to capture this target."""
    for f in my_fleets:
        dist = math.hypot(target.x - f.x, target.y - f.y)
        speed = get_speed(f.ships)
        T = dist / max(1.0, speed)
        
        tx, ty = target.x, target.y
        is_comet = target.id in comet_data
        
        if is_comet:
            c_info = comet_data[target.id]
            arr_idx = c_info["index"] + int(math.ceil(T))
            if arr_idx < len(c_info["path"]):
                tx, ty = c_info["path"][arr_idx]
        else:
            orb_rad = math.hypot(tx - 50, ty - 50)
            if orb_rad + target.radius < 50:
                base_angle = math.atan2(ty - 50, tx - 50)
                new_angle = base_angle + angular_velocity * T
                tx = 50 + orb_rad * math.cos(new_angle)
                ty = 50 + orb_rad * math.sin(new_angle)
                
        angle_to_intercept = math.atan2(ty - f.y, tx - f.x)
        diff = (f.angle - angle_to_intercept + math.pi) % (2 * math.pi) - math.pi
        if abs(diff) < 0.1:
            return True
    return False

def agent(obs):
    moves =[]
    
    # Safely parse Environment Data
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    angular_velocity = obs.get("angular_velocity", 0.0) if isinstance(obs, dict) else obs.angular_velocity
    
    raw_planets = obs.get("planets",[]) if isinstance(obs, dict) else obs.planets
    planets =[Planet(*p) for p in raw_planets]
    
    raw_fleets = obs.get("fleets",[]) if isinstance(obs, dict) else obs.fleets
    fleets =[Fleet(*f) for f in raw_fleets]
    
    # Process Comet Data Paths
    comet_data = {}
    raw_comets = obs.get("comets",[]) if isinstance(obs, dict) else obs.comets
    for c_group in raw_comets:
        p_ids = c_group.get("planet_ids",[])
        paths = c_group.get("paths",[])
        path_index = c_group.get("path_index", 0)
        for i, cid in enumerate(p_ids):
            comet_data[cid] = {"path": paths[i], "index": path_index}
            
    my_planets = [p for p in planets if p.owner == player]
    targets =[p for p in planets if p.owner != player]
    my_fleets =[f for f in fleets if f.owner == player]

    for mine in my_planets:
        # 1. PUMP AND DUMP (Comet Evacuation)
        is_mine_comet = mine.id in comet_data
        if is_mine_comet and mine.ships > 0:
            c_info = comet_data[mine.id]
            remaining_turns = len(c_info["path"]) - c_info["index"]
            
            # If the comet is leaving in less than 20 turns, liquidate all assets
            if remaining_turns < 20:
                best_evac = None
                min_dist = float('inf')
                # Try to evacuate to a permanent planet we own
                for p in my_planets:
                    if p.id not in comet_data and p.id != mine.id:
                        d = math.hypot(p.x - mine.x, p.y - mine.y)
                        if d < min_dist:
                            min_dist = d
                            best_evac = p
                            
                if best_evac:
                    angle = math.atan2(best_evac.y - mine.y, best_evac.x - mine.x)
                    moves.append([mine.id, angle, mine.ships])
                    continue # Assets evacuated, skip to next planet
                    
        # 2. STANDARD TARGET ACQUISITION
        best_target = None
        best_score = -1
        best_cost = 0
        best_angle = 0
        
        for target in targets:
            if is_under_acquisition(target, my_fleets, angular_velocity, comet_data):
                continue
                
            dist = math.hypot(mine.x - target.x, mine.y - target.y)
            base_cost = target.ships + 1
            speed = get_speed(base_cost)
            T = dist / speed
            
            # Predict Enemy Accrued Interest
            if target.owner != -1:
                cost = base_cost + (target.production * math.ceil(T))
                speed = get_speed(cost) 
                T = dist / speed
            else:
                cost = base_cost
                
            if mine.ships < cost:
                continue
                
            # Intercept Trajectory (Iterative path/orbit Prediction)
            tx, ty = target.x, target.y
            is_target_comet = target.id in comet_data
            valid_target = True
            
            for _ in range(3):
                if is_target_comet:
                    c_info = comet_data[target.id]
                    arr_idx = c_info["index"] + int(math.ceil(T))
                    if arr_idx >= len(c_info["path"]):
                        valid_target = False # Comet will have left the board before we arrive
                        break
                    tx, ty = c_info["path"][arr_idx]
                else:
                    orb_rad = math.hypot(target.x - 50, target.y - 50)
                    if orb_rad + target.radius < 50:
                        base_angle = math.atan2(target.y - 50, target.x - 50)
                        new_angle = base_angle + angular_velocity * T
                        tx = 50 + orb_rad * math.cos(new_angle)
                        ty = 50 + orb_rad * math.sin(new_angle)
                
                dist = math.hypot(tx - mine.x, ty - mine.y)
                T = dist / speed
                
            if not valid_target or intersects_sun(mine.x, mine.y, tx, ty):
                continue
                
            # 3. ROI / PROFIT MARGIN EVALUATION
            if is_target_comet:
                c_info = comet_data[target.id]
                arr_idx = c_info["index"] + int(math.ceil(T))
                remaining_life = len(c_info["path"]) - arr_idx
                
                # If we spend 10 ships to gain 5 ships before it disappears, it's a terrible investment
                net_profit = remaining_life - cost
                if net_profit <= 0:
                    continue
                prod_value = net_profit / 10.0 # Scale value for comparison
            else:
                # Permanent enemy targets are worth double because taking it = (+X for us, -X for them)
                prod_value = target.production * (2 if target.owner != -1 else 1)
                
            score = prod_value / (cost + dist * 0.1)
            
            if score > best_score:
                best_score = score
                best_target = target
                best_cost = cost
                best_angle = math.atan2(ty - mine.y, tx - mine.x)
                
        # Execute the trade
        if best_target is not None:
            moves.append([mine.id, best_angle, int(best_cost)])
            mine = Planet(mine.id, mine.owner, mine.x, mine.y, mine.radius, mine.ships - int(best_cost), mine.production)

    return moves

Writing main.py
